# Dry Gas–Gas Mean-Property Rating Test

Demonstrates the v0.5.x iterative mean-property rating entry point,
`BareTubeHeatExchanger.rate(...)`, on the same gas–gas geometry and
boundary conditions as `dry_gas_gas_bare_tube_hx_test.ipynb` (30 °C dry
air in tubes vs. 400 °C wet flue-gas mixture on the outside).

This notebook uses the **real property providers**
(`DryAirPropertyProvider`, `GasMixturePropertyProvider`) and the **real
solver** (`BareTubeHeatExchanger.solve`, via `.rate`) — no notebook-local
correlations.

It contrasts two rating modes on the same inputs:

- **default (`iterate=True`)**: properties re-evaluated at the mean bulk
  temperature on every outer iteration,
- **`iterate=False`**: a single pass with properties evaluated once, at
  the inlet state only.

Scope stays gas-phase / sensible-only: `H2O` in the outside mixture is a
gas-phase component, not condensing moisture. No PsychroLib, no
`MoistAirState`, no dew point, no condensation.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace root:", workspace_root)

In [ ]:
from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle

from core.models.bare_tube import BareTubeHeatExchanger
from core.models.mean_property_rating import RatingSideInput

from core.properties import (
    DryAirPropertyProvider,
    GasMixturePropertyProvider,
    GasMixtureSpec,
)

print("Imports completed.")

## Input Data

Same bundle geometry and mass flows as the v0.4.5 gas–gas test case.

In [ ]:
T0_C = 273.15
p_atm = 101_325.0

def c_to_k(t_c: float) -> float:
    return t_c + T0_C

def kgh_to_kgs(m: float) -> float:
    return m / 3600.0

# --- Cold dry air, inside tubes ---------------------------------------------
inside_T_in_C = 30.0
inside_m_dot_kg_h = 18_220.0

# --- Hot wet flue-gas mixture, outside tubes --------------------------------
# dry-air basis 79/21 with 120 g H2O / kg dry air, as wet mole fractions.
outside_T_in_C = 400.0
outside_m_dot_kg_h = 28_380.0
outside_gas_mixture_components = {
    "N2": 0.6627,
    "O2": 0.1761,
    "H2O": 0.1612,
}

# --- Bare tube bundle geometry -----------------------------------------------
tube_Do = 25.0e-3
tube_wall = 1.5e-3
tube_Di = tube_Do - 2.0 * tube_wall
tube_length = 2.8
tube_k = 50.0  # W/(m*K), carbon steel

tube = BareTube(
    D_i=tube_Di,
    D_o=tube_Do,
    length_total=tube_length,
    length_effective=tube_length,
    wall_k=tube_k,
)

bundle = TubeBundle(
    tube=tube,
    n_rows=36,
    n_tubes_per_row=56,
    pitch_transverse=35.0e-3,
    pitch_longitudinal=35.0e-3,
    layout="staggered",
    n_passes_tube=1,
    flow_arrangement="counterflow",
)

hx = BareTubeHeatExchanger(bundle)

pd.Series({
    "inside_T_in_C": inside_T_in_C,
    "inside_m_dot_kg_h": inside_m_dot_kg_h,
    "outside_T_in_C": outside_T_in_C,
    "outside_m_dot_kg_h": outside_m_dot_kg_h,
    "tube_Do_mm": tube_Do * 1000.0,
    "tube_Di_mm": tube_Di * 1000.0,
    "n_tubes_total": bundle.n_tubes_total,
    "n_rows": bundle.n_rows,
    "flow_arrangement": bundle.flow_arrangement,
}, name="value").to_frame()

## Rating Side Inputs

`RatingSideInput` bundles the property provider with the boundary
conditions for one side. "inside" is the tube side, "outside" is the
bundle side; which side is thermally hot vs. cold is decided from `T_in`.

In [ ]:
inside = RatingSideInput(
    provider=DryAirPropertyProvider(),
    m_dot=kgh_to_kgs(inside_m_dot_kg_h),
    T_in=c_to_k(inside_T_in_C),
    p=p_atm,
)

outside_gas_spec = GasMixtureSpec(
    components=outside_gas_mixture_components,
    basis="mole",
    backend="HEOS",
    imposed_phase="gas",
)

outside = RatingSideInput(
    provider=GasMixturePropertyProvider(outside_gas_spec),
    m_dot=kgh_to_kgs(outside_m_dot_kg_h),
    T_in=c_to_k(outside_T_in_C),
    p=p_atm,
)

print("Rating side inputs prepared.")

## Default Rating: Iterative Mean-Property

In [ ]:
res_mean = hx.rate(inside, outside)

assert res_mean.converged
assert res_mean.iterations > 1

pd.Series({
    "converged": res_mean.converged,
    "iterations": res_mean.iterations,
    "residual_q_rel": res_mean.residual_q_rel,
    "q_kW": res_mean.q / 1e3,
    "UA_W_K": res_mean.UA,
    "U_mean_W_m2K": res_mean.U_mean,
    "T_mean_inside_C": res_mean.T_mean_inside - T0_C,
    "T_mean_outside_C": res_mean.T_mean_outside - T0_C,
    "T_out_inside_C": res_mean.T_out_inside - T0_C,
    "T_out_outside_C": res_mean.T_out_outside - T0_C,
    "inside_v_mean_m_s": res_mean.inside_velocity_mean,
    "outside_v_mean_m_s": res_mean.outside_velocity_mean,
    "inside_Re_mean": res_mean.inside_Re_mean,
    "outside_Re_mean": res_mean.outside_Re_mean,
    "inside_Pr_mean": res_mean.inside_Pr_mean,
    "outside_Pr_mean": res_mean.outside_Pr_mean,
    "inside_alfa_mean_W_m2K": res_mean.inside_alfa_mean,
    "outside_alfa_mean_W_m2K": res_mean.outside_alfa_mean,
    "inside_dp_Pa": res_mean.final_result.tube_side_hydraulic.dp_total,
    "outside_dp_Pa": res_mean.final_result.outside_side_hydraulic.dp_total,
}, name="value").to_frame()

## Comparison: Inlet-Only Single Pass (`iterate=False`)

Same inputs, but properties are evaluated once at the inlet state instead
of at the converged mean bulk state. This is the escape hatch, not the
default — useful here only to quantify how much the large temperature
span (30 → 400 °C) moves the result.

In [ ]:
res_inlet = hx.rate(inside, outside, iterate=False)

assert res_inlet.converged
assert res_inlet.iterations == 1

duty_shift_pct = (res_mean.q - res_inlet.q) / res_inlet.q * 100.0

comparison = pd.DataFrame([
    {
        "mode": "mean-property (default)",
        "iterations": res_mean.iterations,
        "q_kW": res_mean.q / 1e3,
        "UA_W_K": res_mean.UA,
        "T_out_inside_C": res_mean.T_out_inside - T0_C,
        "T_out_outside_C": res_mean.T_out_outside - T0_C,
        "inside_v_m_s": res_mean.inside_velocity_mean,
        "outside_v_m_s": res_mean.outside_velocity_mean,
        "inside_dp_Pa": res_mean.final_result.tube_side_hydraulic.dp_total,
        "outside_dp_Pa": res_mean.final_result.outside_side_hydraulic.dp_total,
    },
    {
        "mode": "inlet-only (iterate=False)",
        "iterations": res_inlet.iterations,
        "q_kW": res_inlet.q / 1e3,
        "UA_W_K": res_inlet.UA,
        "T_out_inside_C": res_inlet.T_out_inside - T0_C,
        "T_out_outside_C": res_inlet.T_out_outside - T0_C,
        "inside_v_m_s": res_inlet.inside_velocity_mean,
        "outside_v_m_s": res_inlet.outside_velocity_mean,
        "inside_dp_Pa": res_inlet.final_result.tube_side_hydraulic.dp_total,
        "outside_dp_Pa": res_inlet.final_result.outside_side_hydraulic.dp_total,
    },
])

print(f"Mean-property vs. inlet-only duty shift: {duty_shift_pct:+.2f} %")
comparison

## Energy-Balance Sanity Check

Checks `q` implied by each side's own `m_dot * cp_mean * dT` against the
reported duty of the converged mean-property rating.

In [ ]:
q_inside = inside.m_dot * res_mean.inside_props_mean.cp * (res_mean.T_out_inside - inside.T_in)
q_outside = outside.m_dot * res_mean.outside_props_mean.cp * (outside.T_in - res_mean.T_out_outside)

print(f"q_inside  = {q_inside / 1e3:.2f} kW")
print(f"q_outside = {q_outside / 1e3:.2f} kW")
print(f"q (rating)= {res_mean.q / 1e3:.2f} kW")

assert abs(q_inside - res_mean.q) / res_mean.q < 0.02
assert abs(q_outside - res_mean.q) / res_mean.q < 0.02

if res_mean.warnings:
    print("\nWARNINGS:")
    for w in res_mean.warnings:
        print(f"  [{w.severity}] {w.code}: {w.message}")
else:
    print("\nNo warnings.")